# Experiment 12: GCG Optimization-Based Attack (E3)

**Reviewer concern (R1, R2):** §14.5 jailbreaks are template-only. R1 asked for at least one optimization-based attack on 2-3 SLMs to test the shallow-alignment hypothesis empirically.

**This notebook:** runs Greedy Coordinate Gradient (GCG) attacks via `nanogcg` on:
- Qwen 2.5-3B-Instruct
- Llama 3.2-3B-Instruct
- Phi-4-mini-instruct

Targets: 50 HarmBench standard test behaviors (the same subset used in §14.5 templates).

Reports per-model ASR with 95% Wilson CI on the GCG-optimized prompts vs. plain prompts vs. the existing §14.5 template attacks.

**Output:** `experiments/results/gcg_attack.json`.

**Runtime:** ~12-18 GPU-hours on A100 (50 behaviors × 3 models × ~7 min/run).


## Setup

In [ ]:
%%capture
!pip install -U 'transformers>=4.51' 'accelerate>=1.1' nanogcg datasets statsmodels -q
import os, json, time, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
import nanogcg
from nanogcg import GCGConfig

print(f'transformers: {transformers.__version__}')
print(f'nanogcg: {nanogcg.__version__ if hasattr(nanogcg, "__version__") else "installed"}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PhD/paper1-survey')
RESULTS_DIR = DRIVE_ROOT / 'experiments' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)


In [ ]:
REPRO = {
    'notebook': '12_gcg_attack',
    'created': datetime.utcnow().isoformat() + 'Z',
    'transformers': transformers.__version__,
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'seed': 42,
    'models': [
        'Qwen/Qwen2.5-3B-Instruct',
        'meta-llama/Llama-3.2-3B-Instruct',
        'microsoft/Phi-4-mini-instruct',
    ],
    'n_behaviors': 50,
    'gcg_num_steps': 250,  # default in nanogcg; reduce to 100 if GPU-budget-constrained
    'gcg_search_width': 512,
    'judge': 'meta-llama/Llama-Guard-3-1B',
}
REPRO


## Load 50 HarmBench behaviors

Use the same 50-behavior subset already used by §14.5 (templated jailbreaks). The list lives in the existing `exp5_jailbreak_*.json` results so we can directly compare GCG ASR vs. template ASR vs. plain ASR on the same behaviors.


In [ ]:
import glob
exp5 = sorted(glob.glob(str(DRIVE_ROOT / 'experiments' / 'results' / 'exp5_*.json')))
if exp5:
    sample = json.loads(Path(exp5[0]).read_text())
    behaviors = sorted({r['behavior_id'] for r in sample.get('results', [])})
    print(f'Loaded {len(behaviors)} behaviors from §14.5 results')
else:
    # Fallback: load HarmBench standard test directly
    from datasets import load_dataset
    ds = load_dataset('walledai/HarmBench', 'standard', split='train')
    behaviors = ds.shuffle(seed=REPRO['seed']).select(range(REPRO['n_behaviors']))
    print(f'Sampled {len(behaviors)} behaviors directly from HarmBench')


## Run GCG per model

We run GCG with default `nanogcg` settings (250 steps, 512 search width). For each (model, behavior), we record: the optimized adversarial suffix, the model's response with the suffix, and the Llama Guard label.


In [ ]:
# TODO[E3]: full attack loop. Skeleton:
#
#   def run_gcg(model_id, behaviors):
#       tok = AutoTokenizer.from_pretrained(model_id)
#       model = AutoModelForCausalLM.from_pretrained(model_id,
#                   torch_dtype=torch.bfloat16, device_map='auto')
#       cfg = GCGConfig(num_steps=REPRO['gcg_num_steps'],
#                       search_width=REPRO['gcg_search_width'],
#                       seed=REPRO['seed'])
#       results = []
#       for b in behaviors:
#           res = nanogcg.run(model, tok, [{'role': 'user', 'content': b['behavior']}],
#                              b['target'], cfg)
#           results.append({'behavior_id': b['BehaviorID'], 'suffix': res.best_string,
#                           'loss': res.best_loss})
#       del model; gc.collect(); torch.cuda.empty_cache()
#       return results
#
# Save per-model partials to gcg_partial_<short>.json for resume support.


## Judge GCG-optimized prompts and compute ASR + Wilson CI

In [ ]:
from statsmodels.stats.proportion import proportion_confint

def asr_with_ci(unsafe_count: int, total: int):
    p = unsafe_count / total if total else 0.0
    if total:
        lo, hi = proportion_confint(unsafe_count, total, alpha=0.05, method='wilson')
    else:
        lo = hi = 0.0
    return p, lo, hi

# TODO[E3]: aggregate gcg_partial_*.json results, judge each generation with Llama Guard 3-1B,
# compare GCG ASR vs §14.5 template ASR vs §14.1 plain ASR per model, save aggregated JSON.
